# Overview

This overview is a short starting point for new users of `fMRI-HA`. We recommend reading it before running the full workflow tutorials. In about 10 minutes, it will help you understand the basic logic of hyperalignment, the type of fMRI data expected by the toolbox, and the main terms used across the documentation.

Hyperalignment can be understood as a three-stage workflow: constructing a functional common space, estimating subject-specific transformations into that space, and applying these transformations to new data. This page introduces these stages at a conceptual level, together with the spatial vocabulary used throughout the manual.

## 1. What fMRI-HA Does

`fMRI-HA` is a toolkit for organizing, running, and evaluating hyperalignment workflows for fMRI data. Hyperalignment addresses the fact that functional response patterns are often not perfectly aligned across individuals, even after anatomical normalization. It learns subject-specific transformations that project each subject's data into a shared functional space, so that common information can be compared and analyzed more directly across subjects.

The workflow in `fMRI-HA` has three core stages. First, subject-level fMRI data are prepared as input arrays, typically one time-by-feature matrix per subject. Second, a functional common space is constructed from training data. Third, subject-specific transformation matrices are estimated and applied to new or held-out data. The resulting aligned data can then be used for downstream evaluation and statistical analysis, including intersubject correlation, decoding or prediction, functional connectivity, and brain--behavior analyses.

`fMRI-HA` supports both Response Hyperalignment (RHA) and Connectivity Hyperalignment (CHA). RHA uses time-locked response patterns to build the common space, whereas CHA uses functional connectivity profiles. In addition to the alignment procedures themselves, the toolbox includes utilities for data preparation, file organization, workflow checking, transformation application, benchmark evaluation, and statistical analysis. Workflows can be run through Python scripts or through the graphical user interface (GUI).

For the core conceptual background, see **Understanding Hyperalignment**.

```{image} pic/overview/work_flow.png
:width: 800px
```

## 2. How fMRI-HA Represents fMRI Data

`fMRI-HA` uses **2D numerical arrays** as the basic input format. In most workflows, each subject is represented by one matrix with the shape `n_samples × n_features`.

Rows are **samples**. The meaning of a sample depends on the analysis. In Response Hyperalignment (RHA), samples are usually time points, such as TRs in a movie-viewing or resting-state run. They can also be trial-level responses, condition estimates, beta maps, or other response units if the data have been prepared that way.

Columns are **spatial features**. For surface-based data, features are usually cortical vertices. For volume-based data, features are usually voxels. For example, one hemisphere of surface-based movie data with 1,200 TRs and 10,242 vertices can be represented as a matrix of shape `1200 × 10242`. In this case, each row is the response pattern at one TR, and each column is the time series of one cortical vertex.

The key requirement is that the matrix structure should be consistent across subjects. When several subjects are used together in one alignment workflow, their rows should refer to comparable samples, and their columns should refer to the same type of spatial features within the selected analysis scope.

The expected matrix shape depends on the alignment type:

- **Response Hyperalignment (RHA)** usually uses a response matrix with the shape `TR × feature`, where rows are time points or response samples and columns are vertices or voxels.
- **Connectivity Hyperalignment (CHA)** follows the same matrix logic, but uses connectivity-profile matrices. The details are introduced later in the CHA tutorial.

In short, before running `fMRI-HA`, users should think of each subject's data as a prepared 2D array: rows define the samples being compared, and columns define the brain features being aligned.

```{image} pic/overview/data_matrix.png
:width: 800px
```

## 3. The Three Core Steps of Hyperalignment

Although **Response Hyperalignment (RHA)** and **Connectivity Hyperalignment (CHA)** use different input matrices, they follow the same basic workflow in `fMRI-HA`:

1. construct a functional common space;
2. estimate subject-specific transformation matrices;
3. apply the transformations to obtain aligned data.

In practical `fMRI-HA` workflows, these three steps are implemented as separate functions. We recommend running them `step by step` in this order, rather than treating hyperalignment as a single black-box operation. This makes the workflow easier to inspect, debug, and reuse across different datasets or downstream analyses.

This section introduces these three steps at a practical conceptual level. For a more detailed explanation of the mathematical logic behind hyperalignment, see **Understanding Hyperalignment**.

### 3.1 Construct a Functional Common Space

The **common space** is a shared functional reference space estimated from training or template data. It is the target space into which subject-specific data will later be projected.

The common space is not simply an anatomical template. An anatomical template defines a shared spatial coordinate system for brain locations. A functional common space defines a shared representational coordinate system for response patterns or connectivity profiles.

In `fMRI-HA`, the common space can be constructed using different algorithms, such as Procrustes-based or PCA-based approaches. The overview does not require users to choose between these methods; the relevant options are introduced in the RHA and CHA workflow pages.

### 3.2 Estimate Subject-Specific T Matrices

After the common space has been constructed, `fMRI-HA` estimates one transformation matrix for each subject. This matrix maps that subject's data from the original subject-specific feature space into the common space.

Throughout the manual, this subject-specific transformation matrix is referred to as a **T matrix**.

When estimating a T matrix, the subject data matrix and the common-space matrix must have compatible shapes. For **RHA**, this usually means that the subject data used for T-matrix estimation must correspond to the same response samples used to define the common space, such as the same movie segment or the same task samples. For example, a common space constructed from the first half of a movie should be used to estimate T matrices from the first half of that movie, not directly from the second half. Once the T matrix has been estimated, it can then be applied to the second half or to other held-out data.

For **CHA**, the same three-stage logic applies, but the matrices are based on functional connectivity profiles. The additional seed-target details are introduced in the CHA workflow page.

### 3.3 Apply T Matrices to Obtain Aligned Data

The final step is to apply each subject's T matrix to the data that should be aligned. In simplified form:

`aligned_data = subject_data @ T_matrix`

Here, `subject_data` is the data matrix to be aligned, `T_matrix` is the transformation estimated for that subject, and `aligned_data` is the transformed data expressed in the common functional space.

After this step, data from different subjects are represented in a comparable functional space. These aligned data can then be used for downstream analyses, such as intersubject correlation, decoding or prediction, representational similarity analysis, and group-level statistics.

In short: the **common space** is the shared target, the **T matrix** is the subject-specific map, and **alignment** is applying that map to data.

```{important}
In most hyperalignment workflows, the data used to estimate transformations should be separated from the data used for final evaluation.

For example, if the same subjects watched one movie run, you may use the first half of the movie to construct the common space and estimate each subject's T matrix, then apply the T matrices to the second half of the movie. This avoids evaluating alignment on the same data that were used to estimate the transformations.

With a larger sample, another option is to use one group of subjects to construct the common space, then estimate T matrices for a separate group of subjects using matched training data, and finally apply these T matrices to held-out or downstream data from the test subjects.

This separation helps reduce overfitting. If the same data are used for common-space construction, T-matrix estimation, and final evaluation, alignment quality metrics such as ISC may be inflated.

In CHA workflows, estimation and application data may be separated in different ways because connectivity profiles can be used to estimate transformations that are later applied to response data or other compatible matrices.
```

## 4. Surface, Volume, and Neuroimaging File Types

The remaining concepts on this page explain the vocabulary used in the workflow tutorials. This section introduces the main spatial data types and file formats without going into preprocessing details.

In fMRI analysis, brain data are commonly represented as either **surface data** or **volume data**. `fMRI-HA` supports both types, as long as they can be converted into 2D arrays with consistent feature order across subjects.

### 4.1 Surface Data

**Surface data** represent cortical signals on a mesh of vertices. Instead of treating the cortex as a 3D voxel grid, surface-based analysis models it as a folded cortical sheet. Each point on this sheet is called a **vertex**.

Surface data are usually organized separately for the left and right hemispheres. In many `fMRI-HA` workflows, the two hemispheres are prepared and aligned separately, using structure names such as `hemi-L` and `hemi-R`.

A surface space defines the number and order of vertices. When surface data are converted into a 2D matrix, this vertex order becomes the column order of the matrix. This order must be consistent across subjects and files within the same analysis.

Surface data are convenient for cortical analyses because local neighborhoods can follow the geometry of the cortical sheet. This is especially useful for searchlight-based hyperalignment.

Surface workflows often involve a target surface space and a cortical or medial-wall mask. These choices are explained in the Data Preparation and RHA tutorials.

### 4.2 Volume Data

**Volume data** represent brain signals as voxels in a 3D space. A **voxel** is a small 3D unit in the brain image, similar to a pixel in a 2D image but with depth.

Although volume data are stored in 3D space, they can be converted into a 2D matrix for hyperalignment. The selected voxels are assigned a fixed order, and this order defines the feature axis of the matrix.

Volume workflows usually operate within a mask or ROI, using structure names such as `volume-mPFC` or other `volume-<roi_name>` labels. Each matrix column corresponds to one selected voxel, and each voxel index can be mapped back to its 3D coordinate for visualization.

As with surface data, the key requirement is consistent feature order across subjects and files within the same analysis.

### 4.3 Common Neuroimaging File Types

Surface and volume data can be stored in different neuroimaging file formats. The file format describes how data are saved on disk, whereas the spatial representation describes how the data are organized for analysis.

Common formats include:

- **GIFTI** (`.gii`): commonly used for cortical surface data, often with separate files for the left and right hemispheres.
- **NIfTI** (`.nii` or `.nii.gz`): commonly used for volume data, such as whole-brain images, masks, or ROI images.
- **CIFTI** (`.dtseries.nii`, etc.): can combine cortical surface data and subcortical volume data in one file. Although CIFTI files also end with `.nii`, the cortical part is typically represented as surface vertices, while subcortical structures are represented as volume voxels.

`fMRI-HA` provides utilities for converting these formats into the 2D arrays required by the alignment functions. If users already have prepared matrices with the expected shape and consistent feature order, these matrices can also be used directly.

```{image} pic/overview/data_type.png
:width: 800px
```

## 5. Searchlights

Searchlight hyperalignment is one of the main workflows supported by `fMRI-HA`. This section introduces the basic idea of a searchlight before the detailed preparation steps are covered in the RHA and CHA tutorials.

A **searchlight** is a local neighborhood around a center feature. It is similar to a small ROI, but instead of using one fixed region, the searchlight moves across the brain. For each center vertex or voxel, nearby features are selected, and hyperalignment is estimated within that local neighborhood.

For **surface data**, a searchlight can be understood as a circular neighborhood on the cortical surface. It is usually defined by a center vertex and a radius along the surface, and includes all vertices within that radius. For **volume data**, a searchlight is defined around a center voxel in 3D space and includes nearby voxels within the chosen radius.

Searchlights are useful because functional correspondence is often local. Estimating one whole-brain transformation matrix can be computationally expensive and may be too coarse for fine-scale cortical patterns. Searchlight hyperalignment instead estimates many local transformations and combines the local results back into a larger cortical or brain-wide representation.

The **searchlight radius** controls the size of the local neighborhood. A smaller radius keeps the alignment more local, but may provide less information for stable estimation. A larger radius includes more features and may improve stability, but reduces spatial specificity and increases computation time.

Because searchlights overlap, the same vertex or voxel can appear in multiple neighborhoods. After local alignment, `fMRI-HA` aggregates these overlapping results to obtain aligned data for the full analysis scope.

Searchlights can also be restricted by masks. Surface searchlights are usually defined within a cortical mask that excludes the medial wall, and volume searchlights can be restricted to a brain mask or another analysis mask. This ensures that local neighborhoods are built only from valid features.

In short, a searchlight defines the local spatial scope used for hyperalignment. Later tutorials explain how to prepare searchlights, choose a radius, and run searchlight-based RHA or CHA workflows in practice.

```{image} pic/overview/searchlight_diagram.png
:width: 800px
```

## 6. Prepared Files, Working Directory, and Naming

`fMRI-HA` uses an organized working directory, usually referred to as `work_dir`, to store prepared data, intermediate files, aligned outputs, logs, and statistical results.

The general rule is:

- **folders describe the processing stage**, such as prepared data, transformation matrices, aligned data, common spaces, logs, or statistical results;
- **filenames describe the data itself**, such as subject, session, run, task, space, hemisphere, ROI, resolution, description, and data subset.

The naming style is **BIDS-like**. Filenames are built from key-value fields separated by underscores. For example:

`sub-01_ses-train_run-01_task-movie_space-surf_hemi-L_roi-cortical_res-onavg32_desc-bold-zscore_set-train.npy`

In this filename, fields such as `sub-01`, `ses-train`, `task-movie`, and `hemi-L` follow the form `key-value`. This makes files easier to inspect manually and easier to retrieve automatically.

Many `fMRI-HA` functions use these fields to search for files. For example, a workflow may select all files with `space-surf`, `hemi-L`, and `set-train`, or distinguish training data from test data using the `set` field. Consistent naming is therefore important for reproducible workflows and for avoiding file selection errors.

The figure below illustrates the basic organization of `work_dir` and how a BIDS-like filename can be read as a set of searchable key-value fields.

```{image} pic/overview/filename.png
:width: 800px
```

## 7. Where to Go Next

After reading this overview, you do not need to understand every implementation detail. The main goal is to know the logic of the workflow and the terms used in later pages.

- For a minimal runnable example, read **Quick Start**.
- For real neuroimaging files, read **Data Preparation** before running alignment.
- For the first complete alignment workflow, start with **Response Hyperalignment (RHA)**.
- For connectivity-based workflows, read **Connectivity Hyperalignment (CHA)** after you are comfortable with the basic RHA logic.
- For mathematical intuition, read **Understanding Hyperalignment**.
- For function-level details, use **API References** as a lookup page.